# [프로젝트: Seq2Seq으로 한국어 번역기 만들기] 코드 구현
## Step 1. 데이터 정제 및 전처리 (Preprocessing)

In [1]:
import os
# TensorFlow C++ 로그 레벨을 3(FATAL)으로 설정하여 불필요한 경고 메시지 출력 차단
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
# 파이썬 경고 메시지 무시
warnings.filterwarnings('ignore')
import logging
# absl 라이브러리(TensorFlow 내부 로그)의 에러 로그만 표시하도록 설정
logging.getLogger('absl').setLevel('ERROR')

import re
import sentencepiece as spm
import torch
from tensorflow.keras.preprocessing.sequence import pad_sequences
from torch.utils.data import TensorDataset, DataLoader

# 1. 텍스트 노이즈 전처리 함수
def preprocess_sentence(sentence, is_english=False):
    sentence = sentence.lower().strip() # 소문자 변환 및 양끝 공백 제거
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence) # 구두점과 단어 사이에 공백 추가
    sentence = re.sub(r'[" "]+', " ", sentence) # 연속된 공백을 하나로 병합
    
    if is_english:
        # 영어: 알파벳, 구두점 이외의 문자 제거
        sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence).strip()
    else:
        # 한국어: 한글, 알파벳, 구두점 이외의 문자 제거
        sentence = re.sub(r"[^ㄱ-ㅎ가-힣a-zA-Z?.!,]+", " ", sentence).strip()
    return sentence

# 2. 데이터 로드 및 중복 제거
with open('./korean-english-park.train/korean-english-park.train.ko', 'r', encoding='utf-8') as f:
    raw_ko = f.read().splitlines()
with open('./korean-english-park.train/korean-english-park.train.en', 'r', encoding='utf-8') as f:
    raw_en = f.read().splitlines()

# zip으로 쌍을 맺고 set으로 중복 제거 후 다시 리스트로 변환
cleaned_corpus = list(set(zip(raw_ko, raw_en)))

# 3. SentencePiece 학습을 위한 텍스트 파일 저장
# SentencePiece는 학습 시 텍스트 파일을 직접 읽으므로, 정제된 문장들을 별도 파일로 저장합니다.
with open('ko_train.txt', 'w', encoding='utf-8') as f_ko, open('en_train.txt', 'w', encoding='utf-8') as f_en:
    for ko, en in cleaned_corpus:
        f_ko.write(preprocess_sentence(ko, is_english=False) + '\n')
        f_en.write(preprocess_sentence(en, is_english=True) + '\n')

# 4. SentencePiece 모델 학습
# vocab_size 10,000개의 단어 조각(Subword)을 생성합니다.
# pad, bos(start), eos(end), unk(unknown) ID를 명시적으로 설정합니다.
VOCAB_SIZE = 10000
print("한국어 SentencePiece 학습 중...")
spm.SentencePieceTrainer.Train(
    f'--input=ko_train.txt --model_prefix=ko_spm --vocab_size={VOCAB_SIZE} '
    f'--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 '
    f'--pad_piece=<pad> --bos_piece=<start> --eos_piece=<end> --unk_piece=<unk>'
)
print("영어 SentencePiece 학습 중...")
spm.SentencePieceTrainer.Train(
    f'--input=en_train.txt --model_prefix=en_spm --vocab_size={VOCAB_SIZE} '
    f'--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 '
    f'--pad_piece=<pad> --bos_piece=<start> --eos_piece=<end> --unk_piece=<unk>'
)

# 5. 훈련된 토크나이저 불러오기
sp_ko = spm.SentencePieceProcessor()
sp_ko.Load('ko_spm.model')
sp_en = spm.SentencePieceProcessor()
sp_en.Load('en_spm.model')

# 6. 문장을 숫자로 인코딩하고 길이 필터링
enc_corpus, dec_corpus = [], []
for ko, en in cleaned_corpus:
    # SentencePiece를 이용해 문장을 숫자 ID 리스트로 변환
    ko_ids = sp_ko.EncodeAsIds(preprocess_sentence(ko, False))
    # 영어는 번역의 시작(1)과 끝(2) 토큰을 강제로 추가
    en_ids = [1] + sp_en.EncodeAsIds(preprocess_sentence(en, True)) + [2]
    
    # 문장 길이가 너무 길면 학습이 어려우므로 40 토큰 이하로 제한
    if len(ko_ids) <= 40 and len(en_ids) <= 40:
        enc_corpus.append(ko_ids)
        dec_corpus.append(en_ids)

print(f"필터링된 최종 데이터 개수: {len(enc_corpus)}")

# 7. 패딩 처리 및 데이터셋/로더 구축
# 길이가 짧은 문장들은 뒤를 0(pad)으로 채워 길이를 40으로 맞춤
enc_tensor = pad_sequences(enc_corpus, padding='post', value=0)
dec_tensor = pad_sequences(dec_corpus, padding='post', value=0)

# 텐서 변환 (모델 입력용)
enc_tensor = torch.tensor(enc_tensor, dtype=torch.long)
dec_tensor = torch.tensor(dec_tensor, dtype=torch.long)

# 딥러닝 모델 학습을 위해 데이터셋과 배치 단위로 섞어주는 DataLoader 생성
dataset = TensorDataset(enc_tensor, dec_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
print("성공적으로 DataLoader가 구축되었습니다!")

I0000 00:00:1782700368.073817    8800 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


한국어 SentencePiece 학습 중...
영어 SentencePiece 학습 중...


sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=ko_train.txt --model_prefix=ko_spm --vocab_size=10000 --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 --pad_piece=<pad> --bos_piece=<start> --eos_piece=<end> --unk_piece=<unk>
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ko_train.txt
  input_format: 
  model_prefix: ko_spm
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentence

필터링된 최종 데이터 개수: 56485
성공적으로 DataLoader가 구축되었습니다!


### Step 2. Attention 기반 Seq2Seq 모델 설계

In [2]:
import torch.nn as nn
import torch.nn.functional as F

# GPU 사용이 가능하다면 cuda, 아니면 cpu를 사용하도록 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBEDDING_DIM = 256   # 단어 벡터의 크기
HIDDEN_SIZE = 512    # 모델 내부 신경망의 은닉 상태(Hidden State) 크기

# 1. Encoder 클래스: 입력 문장(한국어)을 읽고 핵심 의미를 압축함
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim) # 단어 ID를 벡터로 변환
        self.gru = nn.GRU(embedding_dim, hidden_size, batch_first=True) # 문맥 파악을 위한 GRU 레이어
        
    def forward(self, x):
        x = self.embedding(x)
        output, hidden = self.gru(x) # 입력 전체에 대한 output과 마지막 시점의 hidden state 반환
        return output, hidden

# 2. BahdanauAttention 클래스: 번역할 때 입력 문장의 어느 부분에 집중할지 결정
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.W1 = nn.Linear(hidden_size, hidden_size) # 인코더 결과값 변환용 레이어
        self.W2 = nn.Linear(hidden_size, hidden_size) # 디코더 은닉 상태 변환용 레이어
        self.V = nn.Linear(hidden_size, 1) # 점수 계산용 레이어
        
    def forward(self, hidden, enc_output):
        hidden_with_time_axis = hidden.unsqueeze(1) # 차원 맞춤 (batch, 1, hidden)
        
        # 어텐션 스코어 계산: 어떤 단어에 얼마나 집중할지 점수 산출
        score = self.V(torch.tanh(self.W1(enc_output) + self.W2(hidden_with_time_axis)))
        
        # Softmax를 적용하여 가중치 합이 1이 되도록 함 (확률 분포화)
        attention_weights = F.softmax(score, dim=1)
        
        # 가중치가 반영된 context vector 생성 (입력 정보 요약본)
        context_vector = attention_weights * enc_output
        context_vector = torch.sum(context_vector, dim=1)
        return context_vector, attention_weights

# 3. Decoder 클래스: 어텐션을 이용해 한 단어씩 번역문(영어) 생성
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.attention = BahdanauAttention(hidden_size) # 어텐션 레이어 탑재
        # 입력은 임베딩된 단어 + 어텐션 정보를 합친 크기
        self.gru = nn.GRU(embedding_dim + hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size) # 최종 단어 예측용
        
    def forward(self, x, hidden, enc_output):
        # 지금 생성할 단어가 입력 문장의 어떤 부분을 봐야 할지 결정
        context_vector, attention_weights = self.attention(hidden.squeeze(0), enc_output)
        
        x = self.embedding(x)
        # 텍스트 정보와 어텐션 요약본을 붙여서 GRU에 입력
        x = torch.cat([context_vector.unsqueeze(1), x], dim=-1)
        
        # [핵심] 이전 hidden state를 전달하여 기억을 유지하며 단어 생성
        output, hidden = self.gru(x, hidden)
        
        output = output.view(-1, output.size(2))
        x = self.fc(output) # 어떤 단어일지 확률 계산
        return x, hidden, attention_weights

# 모델 객체 생성 후 장치(GPU/CPU)에 배치
encoder = Encoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE).to(device)
decoder = Decoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_SIZE).to(device)

## Step 3. 훈련 및 번역 (Inference)

In [3]:
import torch.optim as optim
import torch.nn as nn

# 옵티마이저(Adam)와 손실 함수(CrossEntropy) 정의
# Adam은 학습률을 자동으로 조절해주어 딥러닝에서 가장 많이 쓰이는 도구입니다.
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.001)
# ignore_index=0은 <pad>(0번 ID)를 오차 계산에서 제외하라는 뜻입니다. (무의미한 0으로 학습 방해 방지)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# 인공지능이 번역하는 함수 (학습이 끝난 후 실제로 결과를 뽑아내는 과정)
def evaluate(sentence):
    encoder.eval() # 모델을 평가 모드(가중치 고정)로 전환
    decoder.eval()
    
    sentence = preprocess_sentence(sentence, is_english=False)
    inputs = sp_ko.EncodeAsIds(sentence) # 문장을 숫자 ID 리스트로 변환
    inputs = torch.tensor(inputs).unsqueeze(0).to(device)
    
    result_ids = []
    with torch.no_grad(): # 기울기 계산을 멈춰 메모리 절약
        enc_out, enc_hidden = encoder(inputs) # 입력 문장을 인코딩하여 요약 정보(hidden) 생성
        dec_hidden = enc_hidden
        dec_input = torch.tensor([[1]]).to(device) # 첫 단어로 <start> 토큰(ID 1) 입력
        
        for t in range(40): # 최대 40단어까지 생성
            # 어텐션을 이용해 현재 시점에 가장 집중해야 할 단어를 결정하여 다음 단어 예측
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_out)
            predicted_id = predictions.argmax(1).item() # 가장 확률이 높은 단어 선택
            
            if predicted_id == 2: # <end> 토큰(ID 2)이 나오면 번역 종료
                break
                
            result_ids.append(predicted_id)
            dec_input = torch.tensor([[predicted_id]]).to(device) # 예측한 단어를 다음 입력으로 사용
            
    return sp_en.DecodeIds(result_ids) # 숫자 ID를 다시 문장으로 복원

# 10회(Epoch) 반복 학습 시작
EPOCHS = 10 

for epoch in range(EPOCHS):
    encoder.train() # 모델을 훈련 모드(가중치 업데이트 가능)로 전환
    decoder.train()
    total_loss = 0
    
    # 데이터로더에서 배치 단위로 데이터를 가져옴
    for batch, (inp, targ) in enumerate(dataloader):
        inp, targ = inp.to(device), targ.to(device)
        loss = 0
        
        enc_output, enc_hidden = encoder(inp)
        dec_hidden = enc_hidden
        dec_input = targ[:, 0].unsqueeze(1) # 디코더의 첫 입력은 <start>
        
        valid_time_steps = 0
        for t in range(1, targ.size(1)):
            # 패딩(0)은 무시하고, 실제 문장 길이에 대해서만 학습
            if targ[:, t].sum() == 0:
                break
                
            # Teacher Forcing: 실제 타겟 데이터를 디코더에 넣어 다음 단어 예측 학습
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            loss += criterion(predictions, targ[:, t]) # 오차 누적
            dec_input = targ[:, t].unsqueeze(1) # 실제 다음 정답을 입력으로 줌
            valid_time_steps += 1
            
        if valid_time_steps > 0:
            batch_loss = (loss / valid_time_steps) # 문장 내 평균 오차 계산
            total_loss += batch_loss.item()
        
            optimizer.zero_grad() # 이전 기울기 초기화
            loss.backward() # 역전파(기울기 계산)
            
            # 그래디언트 클리핑: 기울기가 너무 커져서 nan이 되지 않도록 값을 제한
            torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=1.0)
            
            optimizer.step() # 가중치 업데이트
        
    print(f'Epoch {epoch+1} Loss {total_loss/len(dataloader):.4f}')
    
    # 훈련 중간마다 번역 성능 확인
    sentences = ["오바마는 대통령이다.", "시민들은 도시 속에 산다.", "커피는 필요 없다.", "일곱 명의 사망자가 발생했다."]
    print("--- [SentencePiece 번역 결과] ---")
    for s in sentences:
        print(f"K: {s} \nE: {evaluate(s)}\n")

Epoch 1 Loss 4.8428
--- [SentencePiece 번역 결과] ---
K: 오바마는 대통령이다. 
E: obama is the first time .

K: 시민들은 도시 속에 산다. 
E: the first time in the first round of the world cup , which is the first time in the capital .

K: 커피는 필요 없다. 
E: the dow is not going to be able to be able to be able to be .

K: 일곱 명의 사망자가 발생했다. 
E: the people were killed in the capital , killing at least people were killed .

Epoch 2 Loss 3.8678
--- [SentencePiece 번역 결과] ---
K: 오바마는 대통령이다. 
E: president barack obama is the first time to the president . bush .

K: 시민들은 도시 속에 산다. 
E: the two sides have been killed in the past two years , the report said .

K: 커피는 필요 없다. 
E: it is not clear that it is not clear .

K: 일곱 명의 사망자가 발생했다. 
E: the death toll was killed at least three people dead .

Epoch 3 Loss 3.3501
--- [SentencePiece 번역 결과] ---
K: 오바마는 대통령이다. 
E: he was a president elect . bush .

K: 시민들은 도시 속에 산다. 
E: the two sides are also in the country , where the city was built in the city .

K: 커피는 필요 없다. 
E: the comp

---

# 프로젝트 회고록: Attention 기반 Seq2Seq 번역기 구축 및 고도화

## 1. 프로젝트 개요
* **과제 내용:** 한국어 뉴스를 영어로 번역하는 Attention 기반 Seq2Seq 모델 구축
* **핵심 목표:** 텍스트 데이터 전처리, 토큰화 기법 적용, 모델 설계 및 훈련 프로세스 이해, Loss 값의 안정적 하락 증명

---

## 2. 주요 시행착오 및 오류 해결 과정

### ① 환경 설정 및 라이브러리 호환성 문제
* **시행착오:** 주피터 노트북 및 터미널에서 한국어 형태소 분석기인 `Mecab` 및 `Konlpy` 설치 중 `subprocess-exited-with-error` 에러 발생과 함께 설치 실패.
* **원인 분석:** 최신 파이썬 환경(Python 3.12)의 엄격해진 패키지 버전 표기 규칙과 옛날 방식의 구형 Mecab 패키지 간의 버전 충돌이 원인이었음.
* **해결 과정:** 에러 로그의 `Invalid version` 단서를 바탕으로 구형 패키지 대신 최신 환경과 호환되는 `mecab-python3` 패키지를 터미널에서 새로 설치하고, 주피터 커널을 재시작하여 정상 인식시킴.
* **배운 점:** 개발 환경의 버전 호환성은 프로젝트의 첫 단추이며, 무작정 구글링 코드를 복사하기보다 에러 로그를 읽고 현재 환경에 맞는 패키지를 선택하는 것이 중요함을 깨달음.

### ② 딥러닝 학습 과정에서의 '기울기 폭발 (Loss NaN)'
* **시행착오:** 모델 학습(Step 5)을 시작하자마자 `Loss` 값이 숫자가 아닌 `nan` (Not a Number)으로 출력되며 번역 결과가 `<unk>`로 완전히 무너짐.
* **원인 분석:** 1. **디코더 논리 오류:** 디코더 내 GRU 레이어 연산 시 이전 시점의 은닉 상태(`hidden state`)를 다음 시점으로 제대로 전달하지 않아 문맥 기억이 끊김.
  2. **패딩 데이터 연산 오류:** 배치 내 짧은 문장 뒤에 붙은 무의미한 패딩(0) 데이터 영역까지 무리하게 오차를 계산하려다 '0으로 나누기'와 유사한 연산 폭발 발생.
* **해결 과정:** * 디코더 GRU 코드를 `self.gru(x, hidden)` 형태로 수정하여 문맥 전달력을 복원함.
  * 학습 루프 내에 타겟 문장의 패딩(0)이 시작되면 오차 계산을 즉시 중단하는 필터링 로직 추가.
  * `torch.nn.utils.clip_grad_norm_`를 도입하여 기울기 값의 상한선(안전벨트)을 강제로 제어함.
  * 오염된 가중치를 비우기 위해 인코더/디코더 모델 설계 셀(Step 4)을 재실행하여 백지상태로 새 출발 시킴.
* **배운 점:** 딥러닝 모델은 블랙박스가 아니라 철저한 수학적 연산으로 움직인다는 점을 체감함. 특히 `nan` 발생 시 무작정 재실행하기보다 모델 파라미터 초기화와 데이터 패딩 처리를 먼저 점검해야 한다는 디버깅 자산을 얻음.

### ③ 데이터 부족 및 사전 한계로 인한 성능 저하
* **시행착오:** `Mecab` 기반 토큰화 사용 시, 단어장 크기(10,000개) 제한에 걸려 사전에 포함되지 못한 수많은 일상 단어들이 `<unk>`(Unknown)로 치환되어 번역 품질이 떨어짐.
* **원인 분석:** 약 6만 개의 훈련 데이터 세트 내에 존재하는 희귀 단어들이 단어 기반 토큰화 방식으로는 모두 버려지는 가성비 저하 문제 발생.
* **해결 과정:** 데이터 전처리 방식을 단어 기반에서 구글의 **SentencePiece (Subword Tokenization)** 기법으로 고도화함. 문장을 더 잘게 쪼개어 모르는 단어가 나오더라도 이미 알고 있는 글자 조각(Subword)들의 조합으로 인식하게 만듦.
* **배운 점:** 데이터가 부족하거나 성능 한계에 부딪혔을 때, 대포를 바꾸듯 모델 구조만 탓할 게 아니라 **데이터를 쪼개고 다듬는 전처리(Tokenization) 방식의 변화**가 훨씬 강력하고 현실적인 돌파구가 될 수 있음을 배움.

---

## 3. 프로젝트 총평 및 인사이트
> "이번 프로젝트는 단순히 인공지능 코드를 돌려보는 것을 넘어, **문제를 정의하고 원인을 분석하여 제어하는 엔지니어링의 전 과정**을 경험한 값진 시간이었습니다. 
> 
> 특히 `nan` 폭발을 해결하기 위해 모델 내부의 흐름을 추적하고, 데이터의 한계를 극복하기 위해 SentencePiece라는 서브워드 토큰화 기법을 직접 이식해 보면서 데이터 전처리가 모델 성능에 미치는 절대적인 영향력을 깨달았습니다. 이번에 다진 Seq2Seq와 Attention 메커니즘에 대한 탄탄한 이해를 바탕으로, 대망의 다음 아키텍처인 **트랜스포머(Transformer)** 구조도 더욱 깊이 있게 흡수할 수 있는 자신감이 생겼습니다."